# Lab 2: Clinical Models & the Pipeline API (SOLUTIONS)
## CCI Session 4

**Duration:** 15 minutes

### Clinical Scenario
> An oncologist at KHCC wants to automatically extract medical entities (diseases, drugs, procedures) from clinical notes. You need to evaluate different biomedical NER models and recommend the best one for KHCC's needs.

### Objective
- Use the pipeline() API for clinical NLP tasks
- Understand tokenization for biomedical text
- Compare multiple NER models on the same clinical note
- Run text classification and summarization pipelines

### Setup
Run the cell below to install the required packages.

In [ ]:
# Install required packages
!pip install -q transformers torch pandas

---
### Cell 1: Your First Pipeline (NER)
Named Entity Recognition (NER) extracts structured information from unstructured clinical text. Let's start with a biomedical NER model.

In [ ]:
# === CELL 1: FIRST PIPELINE — NER ===
from transformers import pipeline

clinical_note = """
Patient is a 58-year-old male with newly diagnosed Stage IIIA non-small cell 
lung cancer. Started on carboplatin and paclitaxel combination chemotherapy. 
History of hypertension controlled with lisinopril. Allergic to sulfonamides.
CT scan shows 4.2cm right upper lobe mass with mediastinal lymphadenopathy.
"""

# Create a NER pipeline with the biomedical model
print("Loading biomedical NER model...")
ner = pipeline("ner", model="d4data/biomedical-ner-all", aggregation_strategy="simple")

results = ner(clinical_note)

print("\n=== Extracted Entities ===")
print(f"{'Entity':<35} {'Type':<20} {'Score'}")
print("-" * 65)
for entity in results:
    print(f"{entity['word']:<35} {entity['entity_group']:<20} {entity['score']:.4f}")

---
### Cell 2: Understanding Tokenization
Biomedical models tokenize clinical terms differently than general models. This affects how well a model understands medical vocabulary.

In [ ]:
# === CELL 2: TOKENIZATION ===
from transformers import AutoTokenizer

# Load tokenizers for two different models
tok_bio = AutoTokenizer.from_pretrained("microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract")
tok_gen = AutoTokenizer.from_pretrained("bert-base-uncased")

# Tokenize the word "carboplatin" with both tokenizers
word = "carboplatin"
tokens_bio = tok_bio.tokenize(word)
tokens_gen = tok_gen.tokenize(word)

print(f"=== Tokenization of '{word}' ===")
print(f"  BiomedBERT:  {tokens_bio} ({len(tokens_bio)} tokens)")
print(f"  BERT-base:   {tokens_gen} ({len(tokens_gen)} tokens)")

# Try more clinical terms
clinical_terms = ["paclitaxel", "lymphadenopathy", "sulfonamides", "mediastinal", "lisinopril"]
print("\n=== Clinical Term Tokenization Comparison ===")
print(f"{'Term':<20} {'BiomedBERT tokens':<30} {'BERT-base tokens'}")
print("-" * 80)
for term in clinical_terms:
    bio_tokens = tok_bio.tokenize(term)
    gen_tokens = tok_gen.tokenize(term)
    print(f"{term:<20} {str(bio_tokens):<30} {str(gen_tokens)}")

# Tokenize the full clinical note with both — compare token counts
clinical_note = """
Patient is a 58-year-old male with newly diagnosed Stage IIIA non-small cell 
lung cancer. Started on carboplatin and paclitaxel combination chemotherapy. 
History of hypertension controlled with lisinopril. Allergic to sulfonamides.
CT scan shows 4.2cm right upper lobe mass with mediastinal lymphadenopathy.
"""

bio_encoded = tok_bio(clinical_note)
gen_encoded = tok_gen(clinical_note)
print(f"\n=== Full Note Token Counts ===")
print(f"  BiomedBERT: {len(bio_encoded['input_ids'])} tokens")
print(f"  BERT-base:  {len(gen_encoded['input_ids'])} tokens")
print("\nBiomedical models typically use fewer tokens for clinical text because")
print("medical terms exist as single tokens in their vocabulary.")

---
### Cell 3: Text Classification
Zero-shot classification lets us classify text into categories without any training data — useful for routing clinical notes to the right department.

In [ ]:
# === CELL 3: TEXT CLASSIFICATION ===
from transformers import pipeline

clinical_note = """
Patient is a 58-year-old male with newly diagnosed Stage IIIA non-small cell 
lung cancer. Started on carboplatin and paclitaxel combination chemotherapy. 
History of hypertension controlled with lisinopril. Allergic to sulfonamides.
CT scan shows 4.2cm right upper lobe mass with mediastinal lymphadenopathy.
"""

# Use a zero-shot classification pipeline for clinical note typing
print("Loading zero-shot classifier...")
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

candidate_labels = ["Discharge Summary", "Progress Note", "Consultation", "Pathology Report"]

result = classifier(clinical_note, candidate_labels)

print("\n=== Note Type Classification ===")
for label, score in zip(result["labels"], result["scores"]):
    bar = "█" * int(score * 40)
    print(f"  {label:<25} {score:.4f} {bar}")

print(f"\nPredicted type: {result['labels'][0]} (confidence: {result['scores'][0]:.2%})")

---
### Cell 4: Summarization
Clinical note summarization can save physicians hours of reading time. Let's test a summarization model on a radiology report.

In [ ]:
# === CELL 4: SUMMARIZATION ===
from transformers import pipeline

clinical_note = """
Patient is a 58-year-old male with newly diagnosed Stage IIIA non-small cell 
lung cancer. Started on carboplatin and paclitaxel combination chemotherapy. 
History of hypertension controlled with lisinopril. Allergic to sulfonamides.
CT scan shows 4.2cm right upper lobe mass with mediastinal lymphadenopathy.
"""

# Create a summarization pipeline
print("Loading summarization model...")
summarizer = pipeline("summarization", model="google/flan-t5-small")

summary = summarizer(clinical_note, max_length=60, min_length=15)

print("=== Clinical Note Summarization ===")
print(f"\nOriginal ({len(clinical_note.split())} words):")
print(clinical_note.strip())
print(f"\nSummary:")
print(summary[0]["summary_text"])

# Try with a longer radiology report
radiology_report = """
FINDINGS: The heart size is normal. The mediastinal contours are unremarkable. 
There is a focal consolidation in the right lower lobe measuring approximately 
3.5 cm. Small bilateral pleural effusions are noted. No pneumothorax. 
The osseous structures are intact. A Port-A-Cath is seen with tip in the 
expected position of the SVC.
IMPRESSION: 
"""

rad_summary = summarizer(radiology_report, max_length=60, min_length=10)

print("\n=== Radiology Report Summarization ===")
print(f"\nFindings:")
print(radiology_report.strip())
print(f"\nGenerated Impression:")
print(rad_summary[0]["summary_text"])
print("\nNote: flan-t5-small is a general model. A model fine-tuned on radiology")
print("reports would produce more clinically accurate impressions.")

---
### Cell 5: Compare NER Models
Different NER models may extract different entities from the same text. Let's compare to see which is best for our oncology use case.

In [ ]:
# === CELL 5: MODEL COMPARISON ===
from transformers import pipeline

clinical_note = """
Patient is a 58-year-old male with newly diagnosed Stage IIIA non-small cell 
lung cancer. Started on carboplatin and paclitaxel combination chemotherapy. 
History of hypertension controlled with lisinopril. Allergic to sulfonamides.
CT scan shows 4.2cm right upper lobe mass with mediastinal lymphadenopathy.
"""

# Model 1: d4data/biomedical-ner-all
print("Loading Model 1: d4data/biomedical-ner-all...")
ner1 = pipeline("ner", model="d4data/biomedical-ner-all", aggregation_strategy="simple")
results1 = ner1(clinical_note)

# Model 2: blaze999/Medical-NER
print("Loading Model 2: blaze999/Medical-NER...")
ner2 = pipeline("ner", model="blaze999/Medical-NER", aggregation_strategy="simple")
results2 = ner2(clinical_note)

# Display results side by side
print("\n" + "=" * 70)
print("MODEL COMPARISON: NER Results")
print("=" * 70)

print("\n--- Model 1: d4data/biomedical-ner-all ---")
print(f"{'Entity':<35} {'Type':<20} {'Score'}")
print("-" * 65)
for e in results1:
    print(f"{e['word']:<35} {e['entity_group']:<20} {e['score']:.4f}")

print(f"\nTotal entities found: {len(results1)}")

print("\n--- Model 2: blaze999/Medical-NER ---")
print(f"{'Entity':<35} {'Type':<20} {'Score'}")
print("-" * 65)
for e in results2:
    print(f"{e['word']:<35} {e['entity_group']:<20} {e['score']:.4f}")

print(f"\nTotal entities found: {len(results2)}")

# Summary comparison
entities1 = set(e['word'].strip().lower() for e in results1)
entities2 = set(e['word'].strip().lower() for e in results2)

print("\n--- Comparison Summary ---")
print(f"Only in Model 1: {entities1 - entities2}")
print(f"Only in Model 2: {entities2 - entities1}")
print(f"In both models:  {entities1 & entities2}")

---
### Cell 6: Batch Processing
In real clinical settings, we need to process many notes efficiently. Let's test batch inference.

In [ ]:
# === CELL 6: BATCH INFERENCE ===
from transformers import pipeline
import time

# Create a list of 5 clinical note snippets
notes = [
    "Patient diagnosed with acute myeloid leukemia, started on cytarabine.",
    "Post-mastectomy for invasive ductal carcinoma, receiving tamoxifen.",
    "Stage IV colon cancer with liver metastases, on FOLFOX regimen.",
    "Chronic lymphocytic leukemia managed with ibrutinib 420mg daily.",
    "Thyroid papillary carcinoma status post total thyroidectomy.",
]

ner = pipeline("ner", model="d4data/biomedical-ner-all", aggregation_strategy="simple")

# Batch processing
print("=== Batch Processing ===")
start_batch = time.time()
batch_results = ner(notes)
batch_time = time.time() - start_batch

for i, (note, entities) in enumerate(zip(notes, batch_results)):
    print(f"\nNote {i+1}: {note[:60]}...")
    for e in entities:
        print(f"  -> {e['word']} ({e['entity_group']}, {e['score']:.3f})")

# One-at-a-time processing
start_seq = time.time()
for note in notes:
    _ = ner(note)
seq_time = time.time() - start_seq

print(f"\n=== Timing Comparison ===")
print(f"  Batch processing:       {batch_time:.2f}s")
print(f"  Sequential processing:  {seq_time:.2f}s")
print(f"  Speedup:                {seq_time/batch_time:.1f}x")

---
### Stretch Challenge

Build a simple clinical NER report: extract entities from the 5 notes above and organize them into a **pandas DataFrame** with columns:
- `note_id`
- `entity`
- `entity_type`
- `confidence`

In [ ]:
# Stretch Challenge: Build a clinical NER report DataFrame
import pandas as pd

rows = []
for i, entities in enumerate(batch_results):
    for entity in entities:
        rows.append({
            "note_id": i + 1,
            "entity": entity["word"].strip(),
            "entity_type": entity["entity_group"],
            "confidence": round(entity["score"], 4),
        })

df = pd.DataFrame(rows)

print("=== Clinical NER Report ===")
print(df.to_string(index=False))

print(f"\n=== Summary Statistics ===")
print(f"Total entities extracted: {len(df)}")
print(f"\nEntities by type:")
print(df["entity_type"].value_counts().to_string())

print(f"\nAverage confidence by type:")
print(df.groupby("entity_type")["confidence"].mean().round(4).to_string())

---
### KHCC Connection

This NER pipeline mirrors the entity extraction module being developed for AIDI-DB's clinical note processing system. Key takeaways:

- **Pipeline API** makes it easy to test models without writing boilerplate code
- **Tokenization matters** — biomedical models handle clinical vocabulary more effectively
- **Model comparison** is essential before choosing a model for production
- **Batch processing** is critical for processing thousands of clinical notes efficiently
- All processing happens **locally** — no patient data leaves KHCC's infrastructure